# AI-Powered Autonomous Vehicle Collision Risk Predictor

**Clean Google Colab implementation** — execute cells from top to bottom.

Pipeline: road image + speed → YOLO object detection → Random Forest demonstration label → SQLite logging → optional Gemini report (Google API or OpenRouter) → FastAPI JSON response.

> **Scientific limitation:** the fallback risk classifier uses synthetic rule-generated labels. It is not validated on real crashes. A single image and object count do not provide reliable distance, time-to-collision, or crash probability. This is an educational prototype, not for controlling a real vehicle.

### Before starting
- If you have a previous `vehicle_collision_results.zip`, you can restore it in Cell 2 to reuse your trained weights.
- If no model is found, the notebook downloads lightweight pretrained `yolo11n.pt` weights so the pipeline can run. This fallback is pretrained on COCO and is **not** the same as KITTI fine-tuning.
- For reports, the default provider is OpenRouter with Gemini 3.1 Flash-Lite; it may require credits. Google direct API is also supported.

## Cell 1 — Install dependencies and initialize paths

In [ ]:
%pip install -q ultralytics opencv-python-headless pandas matplotlib scikit-learn joblib fastapi uvicorn python-multipart openai google-genai requests pytest httpx

from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
import os, sys, json, time, socket, shutil, sqlite3, subprocess, zipfile, mimetypes
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import cv2
import torch
from ultralytics import YOLO
from IPython.display import display, Image
from google.colab import files

PROJECT_DIR = Path('/content/vehicle_collision_project')
OUTPUT_DIR = PROJECT_DIR / 'outputs'
TRAIN_DIR = PROJECT_DIR / 'training' / 'kitti_yolo11n'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = PROJECT_DIR / 'vehicle_safety.db'
RISK_MODEL_PATH = OUTPUT_DIR / 'risk_model_demo.joblib'
RESULTS_CSV = TRAIN_DIR / 'results.csv'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

os.environ['VEHICLE_PROJECT_DIR'] = str(PROJECT_DIR)
print('Project:', PROJECT_DIR)
print('Runtime device:', 'GPU' if torch.cuda.is_available() else 'CPU')
print('Existing detector weights:', any(p.exists() for p in [
    OUTPUT_DIR / 'best_vehicle_detector.pt',
    OUTPUT_DIR / 'vehicle_detector.pt',
    TRAIN_DIR / 'weights' / 'best.pt'
]))

## Cell 2 — Optional restore of your previous project archive

In [ ]:
# Set True only if you have the previous vehicle_collision_results.zip downloaded.
RESTORE_PREVIOUS_ARCHIVE = False

if RESTORE_PREVIOUS_ARCHIVE:
    uploaded_archives = files.upload()
    zip_names = [name for name in uploaded_archives if name.lower().endswith('.zip')]
    if not zip_names:
        raise ValueError('No ZIP archive was uploaded.')
    with zipfile.ZipFile(zip_names[0]) as archive:
        # Basic path traversal check before extracting an uploaded archive.
        for member in archive.infolist():
            target = (PROJECT_DIR / member.filename).resolve()
            if target != PROJECT_DIR.resolve() and PROJECT_DIR.resolve() not in target.parents:
                raise ValueError('The archive contains an unsafe path.')
        archive.extractall(PROJECT_DIR)
    print('Restored archive:', zip_names[0])
else:
    print('Skipping archive restoration. Existing files in this runtime will be reused.')

print('Top-level project entries:', [p.name for p in PROJECT_DIR.iterdir()][:30])

## Cell 3 — Load the existing detector; use a pretrained fallback only if necessary

In [ ]:
# Set True only if you explicitly want to try another KITTI fine-tuning run.
# Leave False to avoid unnecessary long training. If training fails, pretrained YOLO weights are used.
TRAIN_ON_KITTI_IF_NO_WEIGHTS = False

trained_candidates = [
    OUTPUT_DIR / 'best_vehicle_detector.pt',
    TRAIN_DIR / 'weights' / 'best.pt',
    OUTPUT_DIR / 'vehicle_detector.pt',
]
MODEL_PATH = next((p for p in trained_candidates if p.is_file()), None)
DETECTOR_SOURCE = 'existing_project_weights' if MODEL_PATH else None

if MODEL_PATH is None and TRAIN_ON_KITTI_IF_NO_WEIGHTS:
    print('No existing weights found. Attempting optional KITTI fine-tuning...')
    try:
        base_model = YOLO('yolo11n.pt')
        training_result = base_model.train(
            data='kitti.yaml',
            epochs=10,
            imgsz=640,
            batch=8 if torch.cuda.is_available() else 4,
            device=DEVICE,
            workers=2,
            project=str(PROJECT_DIR / 'training'),
            name='kitti_yolo11n',
            exist_ok=True,
            seed=42,
        )
        candidate = TRAIN_DIR / 'weights' / 'best.pt'
        if candidate.exists():
            MODEL_PATH = candidate
            DETECTOR_SOURCE = 'fine_tuned_on_kitti'
            RESULTS_CSV = TRAIN_DIR / 'results.csv'
    except Exception as exc:
        print('KITTI training did not complete:', type(exc).__name__)
        print('Will use pretrained YOLO weights if available.')

if MODEL_PATH is None:
    print('Downloading/loading pretrained YOLO11n weights as a fallback...')
    fallback_model = YOLO('yolo11n.pt')
    fallback_source = Path('yolo11n.pt')
    if not fallback_source.exists():
        checkpoint_path = getattr(fallback_model, 'ckpt_path', None)
        if checkpoint_path and Path(checkpoint_path).exists():
            fallback_source = Path(checkpoint_path)
    if fallback_source.exists():
        MODEL_PATH = OUTPUT_DIR / 'vehicle_detector.pt'
        if fallback_source.resolve() != MODEL_PATH.resolve():
            shutil.copy2(fallback_source, MODEL_PATH)
        DETECTOR_SOURCE = 'pretrained_coco_fallback_not_kitti_finetuned'
    else:
        # Still usable in memory, but the API requires a persistent weight file.
        raise FileNotFoundError('YOLO11n checkpoint could not be located after loading.')

if MODEL_PATH is None or not Path(MODEL_PATH).exists():
    raise FileNotFoundError('No usable detector weight file was found.')

# Keep a stable path for the API if the source is a KITTI-trained model.
if DETECTOR_SOURCE in {'existing_project_weights', 'fine_tuned_on_kitti'} and Path(MODEL_PATH).name == 'best.pt':
    stable_model_path = OUTPUT_DIR / 'best_vehicle_detector.pt'
    if Path(MODEL_PATH).resolve() != stable_model_path.resolve():
        shutil.copy2(MODEL_PATH, stable_model_path)
    MODEL_PATH = stable_model_path

metadata = {
    'weights_path': str(MODEL_PATH),
    'source': DETECTOR_SOURCE,
    'class_names': None,
    'trained_for_this_project': DETECTOR_SOURCE == 'fine_tuned_on_kitti',
}
detector = YOLO(str(MODEL_PATH))
metadata['class_names'] = {str(k): str(v) for k, v in detector.names.items()} if isinstance(detector.names, dict) else list(detector.names)
(OUTPUT_DIR / 'detector_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Detector ready:', MODEL_PATH)
print('Detector source:', DETECTOR_SOURCE)
print('Actual supported class names:', detector.names)
print('Training history available:', RESULTS_CSV.exists())

## Cell 4 — Upload/select a road image and run YOLO inference

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def is_source_image(path: Path) -> bool:
    derived_tags = ('_detected', '_integrated', '_annotated')
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS and not any(tag in path.stem.lower() for tag in derived_tags)

source_images = [p for p in OUTPUT_DIR.iterdir() if is_source_image(p)]
if not source_images:
    print('Upload a road or dashboard-camera image (JPG, JPEG, PNG, WEBP, BMP).')
    uploaded_images = files.upload()
    for filename, content in uploaded_images.items():
        if Path(filename).suffix.lower() in IMAGE_EXTENSIONS:
            saved_path = OUTPUT_DIR / Path(filename).name
            saved_path.write_bytes(content)
            source_images.append(saved_path)
if not source_images:
    raise ValueError('No valid road image found or uploaded.')

print('Available images:')
for index, path in enumerate(source_images):
    print(f'{index}: {path.name}')
selected_index = 0  # Change this value to choose another listed image.
TEST_IMAGE_PATH = source_images[selected_index]
print('Using image:', TEST_IMAGE_PATH)

prediction = detector.predict(
    source=str(TEST_IMAGE_PATH), conf=0.25, imgsz=640, device=DEVICE, verbose=False
)[0]

detection_rows = []
class_counts = {}
if prediction.boxes is not None:
    for box in prediction.boxes:
        class_id = int(box.cls.item())
        class_name = str(detector.names[class_id])
        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
        confidence = float(box.conf.item())
        detection_rows.append({
            'class': class_name, 'confidence': round(confidence, 4),
            'x1': round(x1, 2), 'y1': round(y1, 2), 'x2': round(x2, 2), 'y2': round(y2, 2)
        })
        class_counts[class_name] = class_counts.get(class_name, 0) + 1

annotated_path = OUTPUT_DIR / f'{TEST_IMAGE_PATH.stem}_detected.jpg'
if not cv2.imwrite(str(annotated_path), prediction.plot()):
    raise OSError('Could not save the annotated image.')
detections_df = pd.DataFrame(detection_rows, columns=['class', 'confidence', 'x1', 'y1', 'x2', 'y2'])
detections_df.to_csv(OUTPUT_DIR / 'detections.csv', index=False)
(OUTPUT_DIR / 'detections.json').write_text(json.dumps(detection_rows, indent=2), encoding='utf-8')
print('Detected object count:', len(detection_rows))
print('Class counts:', class_counts)
display(detections_df)
display(Image(filename=str(annotated_path)))

## Cell 5 — Download public NGSIM trajectories and train the tabular risk model

This cell downloads up to 500,000 rows from the U.S. Department of Transportation's public NGSIM trajectory dataset, derives surrogate labels from measured time headway, and trains a Random Forest. It does **not** use synthetic training labels. These labels are traffic-safety proxies, not recorded crash outcomes; report performance only as agreement with this surrogate definition.


In [ ]:
# Train a real-data-derived surrogate-risk classifier using public NGSIM trajectories.
# This downloads real vehicle trajectory measurements; labels are surrogate safety labels
# derived from time headway, NOT observed crash/no-crash labels.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib, requests, io, time

NGSIM_CSV_URL = "https://data.transportation.gov/resource/8ect-6jqj.csv"
NGSIM_CACHE_PATH = OUTPUT_DIR / "ngsim_trajectories.csv"
RISK_DATA_PATH = OUTPUT_DIR / "ngsim_surrogate_risk_training.csv"
RISK_MODEL_PATH = OUTPUT_DIR / "risk_model_demo.joblib"
FEATURE_COLUMNS = ["speed_kmh", "obstacle_count"]

def load_ngsim_data():
    if NGSIM_CACHE_PATH.exists() and NGSIM_CACHE_PATH.stat().st_size > 100_000:
        print("Using cached public NGSIM trajectory data:", NGSIM_CACHE_PATH)
        return pd.read_csv(NGSIM_CACHE_PATH, low_memory=False)
    # A bounded download keeps Colab CPU/RAM use manageable. NGSIM is public
    # US Department of Transportation trajectory data, sampled at 10 Hz.
    params = {"$limit": 500000, "$order": "frame_id"}
    response = requests.get(NGSIM_CSV_URL, params=params, timeout=120)
    response.raise_for_status()
    if "json" in response.headers.get("content-type", "").lower():
        raw = pd.DataFrame(response.json())
    else:
        raw = pd.read_csv(io.BytesIO(response.content), low_memory=False)
    if len(raw) < 1000:
        raise RuntimeError(f"NGSIM endpoint returned only {len(raw)} rows; refusing to train on a tiny sample.")
    raw.to_csv(NGSIM_CACHE_PATH, index=False)
    print(f"Downloaded and cached {len(raw):,} NGSIM trajectory rows.")
    return raw

raw_ngsim = load_ngsim_data()
raw_ngsim.columns = [str(c).strip().lower() for c in raw_ngsim.columns]
# Normalize common names across public mirrors/API versions.
aliases = {
    "vehicle_id": ["vehicle_id", "vehicleid"],
    "frame_id": ["frame_id", "frameid"],
    "lane_id": ["lane_id", "laneid"],
    "local_y": ["local_y", "localy"],
    "v_vel": ["v_vel", "v_vel_mph", "speed", "velocity"],
    "time_headway": ["time_headway", "timeheadway"],
}
resolved = {}
for canonical, choices in aliases.items():
    for choice in choices:
        if choice in raw_ngsim.columns:
            resolved[canonical] = choice
            break
missing = [k for k in aliases if k not in resolved]
if missing:
    raise ValueError(
        "NGSIM CSV schema did not contain required columns: " + ", ".join(missing)
        + ". Inspect raw_ngsim.columns and update the aliases before training."
    )

ng = pd.DataFrame({
    "vehicle_id": pd.to_numeric(raw_ngsim[resolved["vehicle_id"]], errors="coerce"),
    "frame_id": pd.to_numeric(raw_ngsim[resolved["frame_id"]], errors="coerce"),
    "lane_id": pd.to_numeric(raw_ngsim[resolved["lane_id"]], errors="coerce"),
    "local_y": pd.to_numeric(raw_ngsim[resolved["local_y"]], errors="coerce"),
    "v_vel_raw": pd.to_numeric(raw_ngsim[resolved["v_vel"]], errors="coerce"),
    "time_headway": pd.to_numeric(raw_ngsim[resolved["time_headway"]], errors="coerce"),
})
ng = ng.dropna(subset=["vehicle_id", "frame_id", "lane_id", "local_y", "v_vel_raw"])
ng = ng[(ng["v_vel_raw"] >= 0) & (ng["v_vel_raw"] <= 150)]
# In the original NGSIM trajectory data, v_Vel is in feet/second.
# Some mirrors may already expose km/h; standard NGSIM values are typically <= 50 ft/s.
ng["speed_kmh"] = ng["v_vel_raw"] * (1.09728 if ng["v_vel_raw"].quantile(.95) < 60 else 1.0)

# Define the surrogate label from measured time headway:
# <=1.0 s: Critical; <=2.0 s: Alert; >2.0 s: Safe.
# These thresholds are operational research proxies, not crash outcomes or universal safety limits.
valid = ng["time_headway"].between(0.05, 10.0)
ng = ng.loc[valid].copy()
ng["risk_level"] = np.select(
    [ng["time_headway"] <= 1.0, ng["time_headway"] <= 2.0],
    ["Critical", "Alert"],
    default="Safe",
)

# Approximate traffic density from vehicles recorded in the same lane and frame.
# local_y is in feet in the original NGSIM schema; 164 ft is approximately 50 m.
ng["obstacle_count"] = 0
for (_, _), idx in ng.groupby(["frame_id", "lane_id"], sort=False).groups.items():
    idx = np.asarray(list(idx), dtype=int)
    y = ng.loc[idx, "local_y"].to_numpy()
    order = np.argsort(y)
    sorted_y = y[order]
    # Count other tracked vehicles within a 50 m longitudinal window.
    left = np.searchsorted(sorted_y, sorted_y - 164.0, side="left")
    right = np.searchsorted(sorted_y, sorted_y + 164.0, side="right")
    counts = right - left - 1
    ng.loc[idx[order], "obstacle_count"] = counts

training_data = ng[["vehicle_id", "frame_id", "speed_kmh", "obstacle_count", "time_headway", "risk_level"]].copy()
training_data = training_data.replace([np.inf, -np.inf], np.nan).dropna()
training_data["obstacle_count"] = training_data["obstacle_count"].clip(0, 30).astype(int)
training_data.to_csv(RISK_DATA_PATH, index=False)

if training_data["risk_level"].nunique() < 3 or len(training_data) < 1000:
    raise RuntimeError("Insufficient valid NGSIM rows/classes for model training. Do not fall back to synthetic labels.")

# Group split by vehicle ID reduces leakage from adjacent frames of the same vehicle.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(training_data, groups=training_data["vehicle_id"]))
X_train = training_data.iloc[train_idx][FEATURE_COLUMNS]
X_test = training_data.iloc[test_idx][FEATURE_COLUMNS]
y_train = training_data.iloc[train_idx]["risk_level"]
y_test = training_data.iloc[test_idx]["risk_level"]

risk_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=250, max_depth=14, min_samples_leaf=4,
        class_weight="balanced_subsample", random_state=42, n_jobs=-1
    )),
])
risk_model.fit(X_train, y_train)
predictions = risk_model.predict(X_test)
print("NGSIM-derived surrogate-label balanced accuracy (NOT crash prediction accuracy):",
      round(balanced_accuracy_score(y_test, predictions), 4))
print(classification_report(y_test, predictions, labels=["Safe", "Alert", "Critical"], zero_division=0))
print("Confusion matrix [Safe, Alert, Critical]:")
print(confusion_matrix(y_test, predictions, labels=["Safe", "Alert", "Critical"]))
joblib.dump(risk_model, RISK_MODEL_PATH)
print("Saved risk model:", RISK_MODEL_PATH)
print("Saved trajectory-derived training table:", RISK_DATA_PATH)
print("Rows:", len(training_data), "| vehicles:", training_data["vehicle_id"].nunique())
print("Label source: measured NGSIM time headway; no synthetic labels used.")
print("Important: this surrogate model is not validated against real crashes.")
example_features = pd.DataFrame([{"speed_kmh": 40.0, "obstacle_count": len(detection_rows)}], columns=FEATURE_COLUMNS)
print("Example experimental surrogate-risk label:", risk_model.predict(example_features)[0])


## Cell 6 — Initialize and verify SQLite

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS scene_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp_utc TEXT NOT NULL DEFAULT '',
            image_name TEXT NOT NULL DEFAULT '',
            speed_kmh REAL NOT NULL DEFAULT 0,
            obstacle_count INTEGER NOT NULL DEFAULT 0,
            class_counts_json TEXT NOT NULL DEFAULT '{}',
            risk_level TEXT NOT NULL DEFAULT 'Unknown',
            detections_json TEXT NOT NULL DEFAULT '[]',
            llm_provider TEXT NOT NULL DEFAULT 'none',
            llm_available INTEGER NOT NULL DEFAULT 0,
            llm_report_json TEXT NOT NULL DEFAULT 'null'
        )
    """)
    present = {row[1] for row in conn.execute('PRAGMA table_info(scene_logs)').fetchall()}
    migrations = {
        'timestamp_utc': "TEXT NOT NULL DEFAULT ''",
        'image_name': "TEXT NOT NULL DEFAULT ''",
        'speed_kmh': 'REAL NOT NULL DEFAULT 0',
        'obstacle_count': 'INTEGER NOT NULL DEFAULT 0',
        'class_counts_json': "TEXT NOT NULL DEFAULT '{}'",
        'risk_level': "TEXT NOT NULL DEFAULT 'Unknown'",
        'detections_json': "TEXT NOT NULL DEFAULT '[]'",
        'llm_provider': "TEXT NOT NULL DEFAULT 'none'",
        'llm_available': 'INTEGER NOT NULL DEFAULT 0',
        'llm_report_json': "TEXT NOT NULL DEFAULT 'null'",
    }
    for column, definition in migrations.items():
        if column not in present:
            conn.execute(f'ALTER TABLE scene_logs ADD COLUMN {column} {definition}')
    schema = conn.execute('PRAGMA table_info(scene_logs)').fetchall()
print('SQLite database initialized:', DB_PATH)
print('Table columns:', [column[1] for column in schema])

## Cell 7 — Configure and test Gemini or OpenRouter

In [ ]:
from openai import OpenAI

# Default to OpenRouter because direct Google AI Studio access may not work in every account.
# Choose: openrouter, google, or none.
LLM_PROVIDER = input('LLM provider [openrouter/google/none] (default openrouter): ').strip().lower() or 'openrouter'
if LLM_PROVIDER not in {'openrouter', 'google', 'none'}:
    raise ValueError('Choose openrouter, google, or none.')
os.environ['LLM_PROVIDER'] = LLM_PROVIDER
os.environ['LLM_ENABLED'] = '0'  # Enable only after the live connection check succeeds.
LLM_TEST_OK = False

if LLM_PROVIDER == 'none':
    print('LLM disabled; CV/ML pipeline will still run.')
elif LLM_PROVIDER == 'openrouter':
    if not os.getenv('OPENROUTER_API_KEY'):
        os.environ['OPENROUTER_API_KEY'] = getpass('Enter OpenRouter API key (hidden): ').strip()
    os.environ['OPENROUTER_MODEL'] = os.getenv('OPENROUTER_MODEL', 'google/gemini-3.1-flash-lite')
    if os.environ.get('OPENROUTER_API_KEY'):
        try:
            llm_client = OpenAI(
                base_url='https://openrouter.ai/api/v1',
                api_key=os.environ['OPENROUTER_API_KEY'],
                timeout=45,
            )
            test_response = llm_client.chat.completions.create(
                model=os.environ['OPENROUTER_MODEL'],
                messages=[{'role': 'user', 'content': 'Reply exactly: API connection successful.'}],
                max_tokens=30,
            )
            print('OpenRouter response:', test_response.choices[0].message.content)
            LLM_TEST_OK = True
        except Exception as exc:
            print('OpenRouter connection test failed:', type(exc).__name__)
            print('Check key, credits, model ID, and internet access. The API will run with LLM reporting disabled.')
else:
    if not os.getenv('GEMINI_API_KEY'):
        os.environ['GEMINI_API_KEY'] = getpass('Enter Google Gemini API key (hidden): ').strip()
    os.environ['GEMINI_MODEL'] = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite')
    if os.environ.get('GEMINI_API_KEY'):
        try:
            from google import genai
            google_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
            test_response = google_client.models.generate_content(
                model=os.environ['GEMINI_MODEL'],
                contents='Reply exactly: API connection successful.'
            )
            print('Gemini response:', test_response.text)
            LLM_TEST_OK = True
        except Exception as exc:
            print('Google Gemini connection test failed:', type(exc).__name__)
            print('Check key, model access, quota, and internet access. The API will run with LLM reporting disabled.')

os.environ['LLM_ENABLED'] = '1' if LLM_TEST_OK else '0'
print('Configured provider:', LLM_PROVIDER)
print('Live LLM test passed:', LLM_TEST_OK)

## Cell 8 — Create the single FastAPI application (`app.py`)

In [ ]:
%%writefile /content/vehicle_collision_project/app.py
"""AI-Powered Autonomous Vehicle Collision Risk Predictor (educational prototype).

The detector and risk model are loaded from artifacts created by the notebook.
This application is not a certified driving-safety system.
"""
from __future__ import annotations

import json
import os
import re
import sqlite3
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import cv2
import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, File, Form, HTTPException, Query, UploadFile

PROJECT_DIR = Path(os.getenv("VEHICLE_PROJECT_DIR", "/content/vehicle_collision_project")).resolve()
OUTPUT_DIR = PROJECT_DIR / "outputs"
DB_PATH = Path(os.getenv("VEHICLE_DB_PATH", str(PROJECT_DIR / "vehicle_safety.db"))).resolve()
RISK_MODEL_PATH = Path(os.getenv("RISK_MODEL_PATH", str(OUTPUT_DIR / "risk_model_demo.joblib"))).resolve()
MAX_IMAGE_BYTES = int(os.getenv("MAX_IMAGE_BYTES", str(15 * 1024 * 1024)))
CONFIDENCE_THRESHOLD = float(os.getenv("YOLO_CONFIDENCE", "0.25"))
IMAGE_SIZE = int(os.getenv("YOLO_IMAGE_SIZE", "640"))
ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _find_detector_weights() -> Path | None:
    candidates = [
        OUTPUT_DIR / "best_vehicle_detector.pt",  # prior notebook artifact
        OUTPUT_DIR / "vehicle_detector.pt",       # generic stable artifact
        PROJECT_DIR / "training" / "kitti_yolo11n" / "weights" / "best.pt",
        OUTPUT_DIR / "yolo11n.pt",
        PROJECT_DIR / "yolo11n.pt",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    return None


# Best-effort model loading allows the module and health endpoint to be inspected
# even when model artifacts/dependencies are absent. The endpoint reports 503 until
# the notebook has prepared the artifacts.
detector: Any = None
risk_model: Any = None
DETECTOR_LOAD_ERROR: str | None = None
RISK_MODEL_LOAD_ERROR: str | None = None
DETECTOR_WEIGHTS_PATH: Path | None = _find_detector_weights()

try:
    if DETECTOR_WEIGHTS_PATH is None:
        raise FileNotFoundError("No YOLO weights found. Run the notebook's model setup cell first.")
    from ultralytics import YOLO

    detector = YOLO(str(DETECTOR_WEIGHTS_PATH))
except Exception as exc:  # health reports unavailability; API won't falsely claim readiness
    DETECTOR_LOAD_ERROR = type(exc).__name__

try:
    if not RISK_MODEL_PATH.is_file():
        raise FileNotFoundError("Risk model is missing. Run the notebook's risk-model cell first.")
    risk_model = joblib.load(RISK_MODEL_PATH)
except Exception as exc:
    RISK_MODEL_LOAD_ERROR = type(exc).__name__

app = FastAPI(
    title="AI-Powered Autonomous Vehicle Collision Risk Predictor",
    description=(
        "Educational prototype combining YOLO object detection, an NGSIM trajectory-derived "
        "surrogate-risk classifier, SQLite logging, and optional Gemini LLM reporting. "
        "Not for controlling or certifying a real vehicle."
    ),
    version="1.2.0",
)


def initialize_database() -> None:
    """Create the log schema and safely add columns used by newer versions."""
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(DB_PATH, timeout=30) as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS scene_logs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp_utc TEXT NOT NULL DEFAULT '',
                image_name TEXT NOT NULL DEFAULT '',
                speed_kmh REAL NOT NULL DEFAULT 0,
                obstacle_count INTEGER NOT NULL DEFAULT 0,
                class_counts_json TEXT NOT NULL DEFAULT '{}',
                risk_level TEXT NOT NULL DEFAULT 'Unknown',
                detections_json TEXT NOT NULL DEFAULT '[]',
                llm_provider TEXT NOT NULL DEFAULT 'none',
                llm_available INTEGER NOT NULL DEFAULT 0,
                llm_report_json TEXT NOT NULL DEFAULT 'null'
            )
            """
        )
        current_columns = {
            row[1] for row in conn.execute("PRAGMA table_info(scene_logs)").fetchall()
        }
        migrations = {
            "timestamp_utc": "TEXT NOT NULL DEFAULT ''",
            "image_name": "TEXT NOT NULL DEFAULT ''",
            "speed_kmh": "REAL NOT NULL DEFAULT 0",
            "obstacle_count": "INTEGER NOT NULL DEFAULT 0",
            "class_counts_json": "TEXT NOT NULL DEFAULT '{}'",
            "risk_level": "TEXT NOT NULL DEFAULT 'Unknown'",
            "detections_json": "TEXT NOT NULL DEFAULT '[]'",
            "llm_provider": "TEXT NOT NULL DEFAULT 'none'",
            "llm_available": "INTEGER NOT NULL DEFAULT 0",
            "llm_report_json": "TEXT NOT NULL DEFAULT 'null'",
        }
        for column, definition in migrations.items():
            if column not in current_columns:
                # Column names/definitions come from this fixed internal map.
                conn.execute(f"ALTER TABLE scene_logs ADD COLUMN {column} {definition}")


initialize_database()


def _json_object_from_text(text: str) -> dict[str, Any]:
    """Parse a JSON object, including when a provider wraps it in Markdown fences."""
    cleaned = (text or "").strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass

    first = cleaned.find("{")
    last = cleaned.rfind("}")
    if first >= 0 and last > first:
        try:
            parsed = json.loads(cleaned[first : last + 1])
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass
    return {"text": cleaned}


def generate_llm_report(
    speed_kmh: float,
    obstacle_count: int,
    class_counts: dict[str, int],
    risk_level: str,
) -> dict[str, Any]:
    """Generate an advisory report with the configured provider; fail gracefully."""
    provider = os.getenv("LLM_PROVIDER", "openrouter").strip().lower()
    enabled = os.getenv("LLM_ENABLED", "1").strip().lower() not in {"0", "false", "no"}

    if provider in {"", "none", "disabled"} or not enabled:
        return {
            "available": False,
            "provider": provider or "none",
            "model": None,
            "report": None,
            "message": "LLM reporting is disabled or its connection test did not pass.",
        }

    prompt = f"""
You are a cautious reporting assistant for an educational vehicle-scene analysis prototype.
Use only the data below. Return one valid JSON object with exactly these keys:
summary (string), observed_facts (array of strings), limitations (array of strings),
cautious_recommendations (array of strings).

Input:
- Vehicle speed: {speed_kmh} km/h
- Detected object count: {obstacle_count}
- Detected class counts: {json.dumps(class_counts, ensure_ascii=False)}
- Experimental ML label: {risk_level}

Critical limitations:
- The risk classifier uses surrogate labels derived from measured NGSIM time headway, not real crash outcomes.
- An image and object count do not establish distance, relative velocity, time-to-collision,
  braking distance, or collision probability.
- Do not invent measurements, declare a collision imminent/certain, or claim certification.
- Recommendations must be general and cautious; do not recommend sudden maneuvers.
- This prototype must never control a real vehicle.
""".strip()

    try:
        if provider == "google":
            api_key = os.getenv("GEMINI_API_KEY", "").strip()
            model_name = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite").strip()
            if not api_key:
                raise ValueError("GEMINI_API_KEY is not configured")
            from google import genai

            client = genai.Client(api_key=api_key)
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
            )
            content = getattr(response, "text", None) or ""
        elif provider == "openrouter":
            api_key = os.getenv("OPENROUTER_API_KEY", "").strip()
            model_name = os.getenv("OPENROUTER_MODEL", "google/gemini-3.1-flash-lite").strip()
            if not api_key:
                raise ValueError("OPENROUTER_API_KEY is not configured")
            from openai import OpenAI

            client = OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key=api_key,
                timeout=45,
                max_retries=1,
            )
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "Return a valid JSON hazard report and obey all stated limitations.",
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.2,
                max_tokens=700,
            )
            content = response.choices[0].message.content or ""
        else:
            raise ValueError("LLM_PROVIDER must be 'google', 'openrouter', or 'none'")

        return {
            "available": True,
            "provider": provider,
            "model": model_name,
            "report": _json_object_from_text(content),
            "message": None,
        }
    except Exception as exc:
        status_code = getattr(exc, "status_code", None)
        safe_message = type(exc).__name__
        if status_code is not None:
            safe_message += f" (HTTP {status_code})"
        # Never return exception details that might include request headers/secrets.
        return {
            "available": False,
            "provider": provider,
            "model": os.getenv(
                "GEMINI_MODEL" if provider == "google" else "OPENROUTER_MODEL",
                "gemini-3.1-flash-lite" if provider == "google" else "google/gemini-3.1-flash-lite",
            ),
            "report": None,
            "message": f"LLM request failed: {safe_message}. Check credentials, model access, quota, and network connectivity.",
        }


@app.get("/")
def root() -> dict[str, str]:
    return {
        "project": "AI-Powered Autonomous Vehicle Collision Risk Predictor",
        "docs": "/docs",
        "health": "/health",
        "analyze_endpoint": "POST /analyze-scene",
        "logs_endpoint": "GET /logs",
    }


@app.get("/health")
def health() -> dict[str, Any]:
    ready = detector is not None and risk_model is not None
    provider = os.getenv("LLM_PROVIDER", "openrouter").strip().lower()
    api_configured = bool(
        os.getenv("GEMINI_API_KEY") if provider == "google" else os.getenv("OPENROUTER_API_KEY")
    ) if provider not in {"none", "disabled", ""} else False
    return {
        "status": "ok" if ready else "degraded",
        "detector_loaded": detector is not None,
        "risk_model_loaded": risk_model is not None,
        "detector_weights": DETECTOR_WEIGHTS_PATH.name if DETECTOR_WEIGHTS_PATH else None,
        "llm_provider": provider or "none",
        "llm_configured": api_configured,
        "llm_enabled": os.getenv("LLM_ENABLED", "1").lower() not in {"0", "false", "no"},
        "risk_model_is_demo": True,
        "risk_model_is_synthetic": False,
        "risk_model_label_source": "NGSIM time-headway surrogate labels; not crash-validated",
    }


@app.post("/analyze-scene")
def analyze_scene(
    image: UploadFile = File(...),
    speed_kmh: float = Form(...),
) -> dict[str, Any]:
    if not np.isfinite(speed_kmh) or not 0 <= speed_kmh <= 300:
        raise HTTPException(status_code=400, detail="speed_kmh must be a finite number between 0 and 300.")

    image_name = Path(image.filename or "road.jpg").name
    suffix = Path(image_name).suffix.lower()
    if suffix not in ALLOWED_EXTENSIONS:
        raise HTTPException(status_code=400, detail="Upload a JPG, JPEG, PNG, WEBP, or BMP image.")

    image_bytes = image.file.read(MAX_IMAGE_BYTES + 1)
    if not image_bytes:
        raise HTTPException(status_code=400, detail="The uploaded image is empty.")
    if len(image_bytes) > MAX_IMAGE_BYTES:
        raise HTTPException(status_code=413, detail=f"Image must be {MAX_IMAGE_BYTES // (1024 * 1024)} MB or smaller.")

    image_array = cv2.imdecode(np.frombuffer(image_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    if image_array is None:
        raise HTTPException(status_code=400, detail="The uploaded file is not a valid decodable image.")

    if detector is None or risk_model is None:
        raise HTTPException(
            status_code=503,
            detail="Models are not ready. Run the notebook model setup cells and check GET /health.",
        )

    try:
        prediction = detector.predict(
            source=image_array,
            conf=CONFIDENCE_THRESHOLD,
            imgsz=IMAGE_SIZE,
            verbose=False,
        )[0]
    except Exception:
        raise HTTPException(status_code=500, detail="YOLO inference failed.") from None

    detections: list[dict[str, Any]] = []
    class_counts: dict[str, int] = {}
    boxes = getattr(prediction, "boxes", None)
    if boxes is not None:
        for box in boxes:
            class_id = int(box.cls.item())
            class_map = getattr(detector, "names", {})
            class_name = str(class_map.get(class_id, class_id) if isinstance(class_map, dict) else class_map[class_id])
            coords = box.xyxy[0].tolist()
            confidence = float(box.conf.item())
            detections.append(
                {
                    "class": class_name,
                    "confidence": round(confidence, 4),
                    "bbox_xyxy": [round(float(value), 2) for value in coords],
                }
            )
            class_counts[class_name] = class_counts.get(class_name, 0) + 1

    # The current tabular model has been trained using total detected-object count.
    obstacle_count = len(detections)
    features = pd.DataFrame(
        [{"speed_kmh": float(speed_kmh), "obstacle_count": int(obstacle_count)}],
        columns=["speed_kmh", "obstacle_count"],
    )
    try:
        risk_level = str(risk_model.predict(features)[0])
    except Exception:
        raise HTTPException(status_code=500, detail="Risk-model prediction failed.") from None

    timestamp = datetime.now(timezone.utc).isoformat()
    time_token = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
    safe_stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(image_name).stem)[:70] or "road"
    annotated_filename = f"{safe_stem}_{time_token}_annotated.jpg"
    annotated_path = OUTPUT_DIR / annotated_filename
    try:
        annotated_image = prediction.plot()
        if not cv2.imwrite(str(annotated_path), annotated_image):
            raise OSError("cv2.imwrite returned false")
    except Exception:
        raise HTTPException(status_code=500, detail="Could not save the annotated image.") from None

    # The report is optional: the CV/ML result and log remain available when the provider fails.
    report = generate_llm_report(speed_kmh, obstacle_count, class_counts, risk_level)
    provider_name = str(report.get("provider", "none"))

    try:
        with sqlite3.connect(DB_PATH, timeout=30) as conn:
            cursor = conn.execute(
                """
                INSERT INTO scene_logs (
                    timestamp_utc, image_name, speed_kmh, obstacle_count,
                    class_counts_json, risk_level, detections_json,
                    llm_provider, llm_available, llm_report_json
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    timestamp,
                    image_name,
                    float(speed_kmh),
                    int(obstacle_count),
                    json.dumps(class_counts, ensure_ascii=False),
                    risk_level,
                    json.dumps(detections, ensure_ascii=False),
                    provider_name,
                    int(bool(report.get("available"))),
                    json.dumps(report.get("report"), ensure_ascii=False),
                ),
            )
            request_id = cursor.lastrowid
    except sqlite3.Error:
        raise HTTPException(status_code=500, detail="Could not save the scene analysis to SQLite.") from None

    return {
        "request_id": request_id,
        "timestamp_utc": timestamp,
        "input": {"image_name": image_name, "speed_kmh": float(speed_kmh)},
        "computer_vision": {
            "detected_object_count": obstacle_count,
            "class_counts": class_counts,
            "detections": detections,
            "annotated_image_filename": annotated_filename,
        },
        "risk_prediction": {
            "label": risk_level,
            "model": "RandomForest trained on NGSIM trajectory-derived surrogate labels",
            "validated_on_real_crashes": False,
        },
        "gemini_copilot": report,
        "limitations": [
            "The risk model uses NGSIM time-headway surrogate labels, not real crash outcomes.",
            "Object count is not distance or collision probability.",
            "A single image does not establish reliable distance or time-to-collision.",
            "This prototype is not certified and must not control a real vehicle.",
        ],
    }


@app.get("/logs")
def get_logs(limit: int = Query(default=20, ge=1, le=100)) -> dict[str, Any]:
    with sqlite3.connect(DB_PATH, timeout=30) as conn:
        conn.row_factory = sqlite3.Row
        records = conn.execute(
            "SELECT * FROM scene_logs ORDER BY id DESC LIMIT ?", (limit,)
        ).fetchall()
    return {"count": len(records), "logs": [dict(record) for record in records]}


## Cell 9 — Write requirements, safe environment template, README, and tests

In [ ]:
%%writefile /content/vehicle_collision_project/requirements.txt
ultralytics>=8.3.0
opencv-python-headless>=4.10.0
numpy>=1.26
pandas>=2.0
scikit-learn>=1.4
joblib>=1.3
fastapi>=0.115
uvicorn[standard]>=0.30
python-multipart>=0.0.9
requests>=2.31
google-genai>=1.0
openai>=1.50
pytest>=8.0
httpx>=0.27


In [ ]:
%%writefile /content/vehicle_collision_project/.env.example
# Select: google, openrouter, or none
LLM_PROVIDER=openrouter
LLM_ENABLED=1

# Direct Google Gemini API
GEMINI_API_KEY=
GEMINI_MODEL=gemini-3.1-flash-lite

# OpenRouter-compatible API (Gemini model routed through OpenRouter)
OPENROUTER_API_KEY=
OPENROUTER_MODEL=google/gemini-3.1-flash-lite

# Optional application settings
VEHICLE_PROJECT_DIR=/content/vehicle_collision_project
YOLO_CONFIDENCE=0.25
YOLO_IMAGE_SIZE=640
MAX_IMAGE_BYTES=15728640


In [ ]:
%%writefile /content/vehicle_collision_project/README.md
# AI-Powered Autonomous Vehicle Collision Risk Predictor

Educational prototype combining image-based object detection, experimental tabular ML, an optional Gemini report, a SQLite log, and FastAPI. **It is not a certified collision-warning or autonomous-driving system.**

## Architecture

```mermaid
flowchart LR
    A[Road image + speed telemetry] --> B[YOLO object detector]
    B --> C[Object counts and bounding boxes]
    C --> D[Random Forest NGSIM-derived surrogate-risk classifier]
    A --> E[SQLite scene log]
    D --> E
    C --> F[Gemini report through Google API or OpenRouter]
    D --> F
    E --> G[Unified JSON response]
    F --> G
```

## Important model limitation

The risk classifier is trained on public NGSIM vehicle trajectories and uses measured time-headway thresholds to create surrogate safety labels. It is not trained on observed crash/no-crash outcomes; metrics measure agreement with those surrogate labels and are **not** real crash prediction accuracy. A single image and the number of visible objects do not establish object distance, relative velocity, time-to-collision, or collision probability. Do not use this prototype to control a real vehicle.

YOLO classes depend on the actual model weights. A pretrained COCO model does not necessarily recognize cones or barriers. The notebook prints the loaded class names.

## Google Colab (recommended)

1. Open `AI_Powered_Vehicle_Collision_Risk_Predictor_Clean.ipynb` in [Google Colab](https://colab.research.google.com/) using **File → Upload notebook**.
2. Run cells in order, from top to bottom. The notebook checks for existing detector weights first. If none are available, it downloads lightweight pretrained `yolo11n.pt` weights; this fallback is not KITTI fine-tuning.
3. Upload a road image if the notebook cannot find one under `outputs/`.
4. Configure the LLM provider in the credential cell. The default is OpenRouter; direct Google Gemini is also supported. Keys are entered with `getpass` and should never be committed to source code.
5. Run the FastAPI startup cell, then the end-to-end test cell.
6. The final cell exports `outputs/scene_logs.csv` and creates `/content/vehicle_collision_results.zip` for download.

### Reusing the previously trained model

If you have a previous `vehicle_collision_results.zip`, upload it into Colab and extract it into `/content/vehicle_collision_project` before model setup. The notebook reuses `outputs/best_vehicle_detector.pt` or `training/kitti_yolo11n/weights/best.pt` when present.

## Local setup

Use Python 3.10–3.12 in a virtual environment, install packages, provide compatible model artifacts in the expected locations, then start from this folder:

```bash
pip install -r requirements.txt
export VEHICLE_PROJECT_DIR="$(pwd)"
export LLM_PROVIDER=none
uvicorn app:app --host 127.0.0.1 --port 8000
```

Windows PowerShell environment variables can be set with `$env:VEHICLE_PROJECT_DIR=(Get-Location).Path` and `$env:LLM_PROVIDER="none"`.

The API loads these artifacts if present:
- `outputs/best_vehicle_detector.pt` (preferred)
- `outputs/vehicle_detector.pt`
- `training/kitti_yolo11n/weights/best.pt`
- `outputs/yolo11n.pt` or `yolo11n.pt`
- `outputs/risk_model_demo.joblib`

If model weights or the risk model are missing, `/health` reports a degraded status and `/analyze-scene` returns HTTP 503 until you prepare the artifacts using the notebook cells.

## API

- `GET /` – project and route overview
- `GET /health` – readiness and LLM configuration (never returns secrets)
- `POST /analyze-scene` – multipart fields `image` and `speed_kmh`
- `GET /logs?limit=20` – latest log records (limit from 1 to 100)

Example request:

```bash
curl -X POST "http://127.0.0.1:8000/analyze-scene" \
  -F "image=@road.jpg" \
  -F "speed_kmh=40"
```

The response contains `computer_vision`, `risk_prediction`, `gemini_copilot`, and `limitations`. A successful HTTP request does not by itself mean the LLM report succeeded; check `gemini_copilot.available`.

## LLM credentials

Set `LLM_PROVIDER=google` and `GEMINI_API_KEY` for the official Google Gen AI SDK, or set `LLM_PROVIDER=openrouter`, `OPENROUTER_API_KEY`, and `OPENROUTER_MODEL` for OpenRouter. Use `LLM_PROVIDER=none` for offline testing. The notebook accepts secrets interactively using `getpass`; do not place live keys in a notebook or ZIP archive.

OpenRouter model identifiers and provider availability can change. Verify the selected model at https://openrouter.ai/models. API usage may incur cost or require credits; do not assume a model is free.

## Tests

Automated API tests mock the detector, risk model, and LLM; they do not require model weights, real API credentials, or paid LLM access.

```bash
pytest -q
```

A separate live test with actual YOLO weights, a real road image, and valid LLM credentials is required to validate the full external integration in your runtime.

## Output files

- Annotated images: `outputs/*_annotated.jpg`
- Detection CSV/JSON: `outputs/detections.csv`, `outputs/detections.json`
- Database: `vehicle_safety.db`
- Exported database logs: `outputs/scene_logs.csv`
- Final Colab archive: `/content/vehicle_collision_results.zip`


## Public trajectory data source

The risk-model training cell downloads NGSIM US-101 trajectory data from the U.S. Department of Transportation public data endpoint: https://data.transportation.gov/resource/8ect-6jqj.csv. Labels are derived from measured time headway as surrogate safety categories; they are not observed crashes. Review the dataset metadata at https://catalog.data.gov/dataset/next-generation-simulation-ngsim-vehicle-trajectories-and-supporting-data.


In [ ]:
%%writefile /content/vehicle_collision_project/tests/test_api.py
"""Mocked API tests; they do not require real YOLO weights or paid LLM credits."""
from __future__ import annotations

import os
import tempfile
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
TEST_PROJECT = Path(tempfile.mkdtemp(prefix="vehicle_collision_test_"))
os.environ["VEHICLE_PROJECT_DIR"] = str(TEST_PROJECT)
os.environ["VEHICLE_DB_PATH"] = str(TEST_PROJECT / "vehicle_safety.db")
os.environ["RISK_MODEL_PATH"] = str(TEST_PROJECT / "outputs" / "risk_model_demo.joblib")
os.environ["LLM_PROVIDER"] = "none"
os.environ["LLM_ENABLED"] = "0"

import cv2
import numpy as np
import pytest
from fastapi.testclient import TestClient

import app as app_module


class FakeBox:
    cls = np.array(2)
    conf = np.array(0.91)
    xyxy = np.array([[10.0, 12.0, 42.0, 35.0]])


class FakePrediction:
    boxes = [FakeBox()]

    @staticmethod
    def plot():
        return np.zeros((48, 64, 3), dtype=np.uint8)


class FakeDetector:
    names = {2: "car"}

    @staticmethod
    def predict(**kwargs):
        return [FakePrediction()]


class FakeRiskModel:
    @staticmethod
    def predict(frame):
        assert list(frame.columns) == ["speed_kmh", "obstacle_count"]
        assert frame.iloc[0]["speed_kmh"] == 40.0
        assert frame.iloc[0]["obstacle_count"] == 1
        return np.array(["Alert"])


@pytest.fixture()
def client(monkeypatch):
    app_module.detector = FakeDetector()
    app_module.risk_model = FakeRiskModel()
    monkeypatch.setattr(
        app_module,
        "generate_llm_report",
        lambda *args: {
            "available": False,
            "provider": "none",
            "model": None,
            "report": None,
            "message": "Disabled in mocked test.",
        },
    )
    return TestClient(app_module.app)


def png_bytes() -> bytes:
    image = np.zeros((48, 64, 3), dtype=np.uint8)
    ok, encoded = cv2.imencode(".png", image)
    assert ok
    return encoded.tobytes()


def test_health_endpoint(client):
    response = client.get("/health")
    assert response.status_code == 200
    payload = response.json()
    assert payload["status"] == "ok"
    assert payload["detector_loaded"] is True
    assert payload["risk_model_loaded"] is True


def test_analyze_scene_end_to_end_and_database(client):
    response = client.post(
        "/analyze-scene",
        files={"image": ("road.png", png_bytes(), "image/png")},
        data={"speed_kmh": "40"},
    )
    assert response.status_code == 200, response.text
    payload = response.json()
    assert payload["computer_vision"]["detected_object_count"] == 1
    assert payload["computer_vision"]["class_counts"] == {"car": 1}
    assert payload["risk_prediction"]["label"] == "Alert"
    assert payload["gemini_copilot"]["available"] is False
    assert payload["risk_prediction"]["validated_on_real_crashes"] is False

    image_output = app_module.OUTPUT_DIR / payload["computer_vision"]["annotated_image_filename"]
    assert image_output.is_file()

    logs = client.get("/logs?limit=5")
    assert logs.status_code == 200
    assert logs.json()["count"] == 1
    saved_record = logs.json()["logs"][0]
    assert saved_record["speed_kmh"] == 40.0
    assert saved_record["obstacle_count"] == 1
    assert saved_record["risk_level"] == "Alert"


def test_rejects_invalid_speed(client):
    response = client.post(
        "/analyze-scene",
        files={"image": ("road.png", png_bytes(), "image/png")},
        data={"speed_kmh": "999"},
    )
    assert response.status_code == 400


def test_rejects_unsupported_extension(client):
    response = client.post(
        "/analyze-scene",
        files={"image": ("data.csv", b"a,b\n1,2", "text/csv")},
        data={"speed_kmh": "40"},
    )
    assert response.status_code == 400


def test_rejects_non_image_payload(client):
    response = client.post(
        "/analyze-scene",
        files={"image": ("road.png", b"not an image", "image/png")},
        data={"speed_kmh": "40"},
    )
    assert response.status_code == 400


def test_logs_limit_is_validated(client):
    assert client.get("/logs?limit=0").status_code == 422


def test_llm_disabled_returns_explicit_status(monkeypatch):
    monkeypatch.setenv("LLM_PROVIDER", "none")
    monkeypatch.setenv("LLM_ENABLED", "0")
    report = app_module.generate_llm_report(40.0, 1, {"car": 1}, "Alert")
    assert report["available"] is False
    assert report["report"] is None
    assert "disabled" in report["message"].lower()


def test_json_parser_handles_markdown_fences():
    parsed = app_module._json_object_from_text('```json\n{"summary": "ok"}\n```')
    assert parsed == {"summary": "ok"}


def test_model_unavailable_returns_503(monkeypatch, client):
    app_module.detector = None
    app_module.risk_model = None
    response = client.post(
        "/analyze-scene",
        files={"image": ("road.png", png_bytes(), "image/png")},
        data={"speed_kmh": "40"},
    )
    assert response.status_code == 503


## Cell 10 — Start FastAPI and verify readiness

In [ ]:
# Ensure all artifacts and credentials are ready before launching the process.
os.environ['VEHICLE_PROJECT_DIR'] = str(PROJECT_DIR)
assert MODEL_PATH.exists(), f'Detector weights missing: {MODEL_PATH}'
assert RISK_MODEL_PATH.exists(), f'Risk model missing: {RISK_MODEL_PATH}'

if 'server' in globals() and server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server.kill()

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.bind(('127.0.0.1', 0))
    API_PORT = sock.getsockname()[1]
API_ROOT = f'http://127.0.0.1:{API_PORT}'
API_LOG_PATH = Path('/content/vehicle_api.log')
api_log_handle = open(API_LOG_PATH, 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', str(API_PORT)],
    cwd=str(PROJECT_DIR), stdout=api_log_handle, stderr=subprocess.STDOUT, env=os.environ.copy()
)

for attempt in range(30):
    try:
        health_response = requests.get(f'{API_ROOT}/health', timeout=3)
        if health_response.ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    print(API_LOG_PATH.read_text(errors='replace'))
    raise RuntimeError('FastAPI did not start; inspect the log above.')

health = health_response.json()
print('Health status:', health_response.status_code)
print(json.dumps(health, indent=2, ensure_ascii=False))
if not health.get('detector_loaded') or not health.get('risk_model_loaded'):
    raise RuntimeError('API process started but one or more models did not load. Review the server log.')
print('API docs:', f'{API_ROOT}/docs')

## Cell 11 — End-to-end test using image + telemetry

In [ ]:
speed_kmh = float(input('Enter test speed in km/h (for example, 40): '))
if not np.isfinite(speed_kmh) or not 0 <= speed_kmh <= 300:
    raise ValueError('Speed must be a finite value between 0 and 300 km/h.')

mime_type = mimetypes.guess_type(TEST_IMAGE_PATH.name)[0] or 'application/octet-stream'
with open(TEST_IMAGE_PATH, 'rb') as image_file:
    api_response = requests.post(
        f'{API_ROOT}/analyze-scene',
        files={'image': (TEST_IMAGE_PATH.name, image_file, mime_type)},
        data={'speed_kmh': str(speed_kmh)},
        timeout=180,
    )

print('HTTP status:', api_response.status_code)
api_result = api_response.json()
if not api_response.ok:
    print(json.dumps(api_result, indent=2, ensure_ascii=False))
    raise RuntimeError('End-to-end API request failed.')

print('Computer vision:')
print(json.dumps(api_result['computer_vision'], indent=2, ensure_ascii=False))
print('Risk prediction:')
print(json.dumps(api_result['risk_prediction'], indent=2, ensure_ascii=False))
print('LLM status/report:')
print(json.dumps(api_result['gemini_copilot'], indent=2, ensure_ascii=False))

api_annotated_path = OUTPUT_DIR / api_result['computer_vision']['annotated_image_filename']
assert api_annotated_path.is_file(), 'Annotated image was not created.'
display(Image(filename=str(api_annotated_path)))
assert api_result['request_id'] is not None
print('End-to-end CV + ML + SQLite request succeeded. LLM available:', api_result['gemini_copilot']['available'])

## Cell 12 — Verify logs, run tests, and package the project

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    logs_df = pd.read_sql_query('SELECT * FROM scene_logs ORDER BY id DESC LIMIT 100', conn)
logs_csv_path = OUTPUT_DIR / 'scene_logs.csv'
logs_df.to_csv(logs_csv_path, index=False)
print('SQLite rows exported:', len(logs_df))
print('CSV:', logs_csv_path)
display(logs_df.head(10))

# Run mocked API tests; these should not need model weights or paid LLM calls.
print('Running automated API tests...')
test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', 'tests/test_api.py'],
    cwd=str(PROJECT_DIR), text=True, capture_output=True,
    env=os.environ.copy(),
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr[-4000:])
if test_run.returncode != 0:
    raise RuntimeError('Automated tests failed; fix the reported test error before packaging.')

# Keep deliverables, but exclude caches and any accidental real .env credential file.
archive_path = Path('/content/vehicle_collision_results.zip')
if archive_path.exists():
    archive_path.unlink()
exclude_parts = {'__pycache__', '.pytest_cache', '.ipynb_checkpoints', '.git'}
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT_DIR.rglob('*'):
        if not path.is_file():
            continue
        if any(part in exclude_parts for part in path.parts):
            continue
        if path.name == '.env' or path.suffix in {'.pyc', '.log'}:
            continue
        archive.write(path, arcname=path.relative_to(PROJECT_DIR.parent))

print('Project archive created:', archive_path)
print('Annotated image:', api_annotated_path)
print('Database:', DB_PATH)
print('CSV log:', logs_csv_path)
print('LLM reporting succeeded in live request:', api_result['gemini_copilot']['available'])
files.download(str(archive_path))